# aloevera — `avr.plot()` Demo

`avr.plot(kind, ...)` is a thin wrapper around `plotly.express` that applies
a consistent default layout (`plotly_white` template, 400 × 700, centred title).

Pass any `plotly.express` function name as `kind` — all remaining arguments
are forwarded unchanged.  Override any default per-call, or change
`avr.DEFAULT_LAYOUT` to update the global baseline.

In [ ]:
import aloevera as avr
import pandas as pd
import numpy as np

print(f"aloevera {avr.__version__}")
print("DEFAULT_LAYOUT:", avr.DEFAULT_LAYOUT)

## 1. Sample data

In [ ]:
rng = np.random.default_rng(42)

# Monthly sales
months = ["Jan","Feb","Mar","Apr","May","Jun",
          "Jul","Aug","Sep","Oct","Nov","Dec"]
df_sales = pd.DataFrame({
    "month":     months,
    "revenue":   rng.integers(50_000, 150_000, 12).tolist(),
    "expenses":  rng.integers(30_000, 100_000, 12).tolist(),
    "profit":    rng.integers(10_000, 60_000,  12).tolist(),
    "customers": rng.integers(200, 800, 12).tolist(),
})

# 90-day time series
df_ts = pd.DataFrame({
    "day":    list(range(1, 91)),
    "price":  np.cumsum(rng.standard_normal(90) * 2 + 0.1).tolist(),
    "volume": rng.integers(100, 1000, 90).tolist(),
})

# Distribution data — three groups
df_dist = pd.DataFrame({
    "value": np.concatenate([
        rng.normal(50, 10, 200),
        rng.normal(65, 12, 200),
        rng.normal(45,  8, 200),
    ]),
    "group": ["A"] * 200 + ["B"] * 200 + ["C"] * 200,
})

# Product category performance
df_cat = pd.DataFrame({
    "category": ["Electronics", "Clothing", "Food", "Books", "Sports", "Home"],
    "q1": rng.integers(10_000, 80_000, 6).tolist(),
    "q2": rng.integers(10_000, 80_000, 6).tolist(),
    "q3": rng.integers(10_000, 80_000, 6).tolist(),
    "q4": rng.integers(10_000, 80_000, 6).tolist(),
})

# Correlation matrix
df_corr = df_sales[["revenue", "expenses", "profit", "customers"]].corr().reset_index()
df_corr_long = df_corr.melt(id_vars="index", var_name="variable", value_name="correlation")

print("df_sales:",  df_sales.shape)
print("df_ts:",     df_ts.shape)
print("df_dist:",   df_dist.shape)
print("df_cat:",    df_cat.shape)

## 2. Line chart — `kind='line'`

In [ ]:
avr.plot(
    'line', df_ts,
    x='day', y='price',
    title='90-Day Price Trend',
    labels={'day': 'Day', 'price': 'Price ($)'},
    line_shape='spline',
)

## 3. Bar chart — `kind='bar'`

In [ ]:
avr.plot(
    'bar', df_sales,
    x='month', y=['revenue', 'expenses'],
    title='Monthly Revenue vs Expenses',
    barmode='group',
    labels={'value': 'Amount ($)', 'variable': 'Metric'},
)

## 4. Scatter plot — `kind='scatter'`

In [ ]:
avr.plot(
    'scatter', df_sales,
    x='customers', y='revenue',
    size='profit', text='month',
    title='Customers vs Revenue (bubble = profit)',
    labels={'customers': 'Customers', 'revenue': 'Revenue ($)'},
)

## 5. Histogram — `kind='histogram'`

In [ ]:
avr.plot(
    'histogram', df_dist,
    x='value', color='group',
    barmode='overlay', opacity=0.7,
    nbins=40,
    title='Value Distribution by Group',
)

## 6. Box plot — `kind='box'`

In [ ]:
avr.plot(
    'box', df_dist,
    x='group', y='value',
    color='group',
    title='Value Distribution — Box Plot',
    points='outliers',
)

## 7. Violin plot — `kind='violin'`

In [ ]:
avr.plot(
    'violin', df_dist,
    x='group', y='value',
    color='group', box=True,
    title='Value Distribution — Violin Plot',
)

## 8. Area chart — `kind='area'`

In [ ]:
# Melt to long form for stacked area
df_area = df_sales.melt(id_vars='month', value_vars=['revenue', 'expenses', 'profit'],
                         var_name='metric', value_name='amount')

avr.plot(
    'area', df_area,
    x='month', y='amount', color='metric',
    title='Monthly Financials — Stacked Area',
    labels={'amount': 'Amount ($)', 'metric': 'Metric'},
)

## 9. Pie chart — `kind='pie'`

In [ ]:
avr.plot(
    'pie', df_cat,
    names='category', values='q1',
    title='Q1 Revenue by Category',
    hole=0.3,   # donut style
)

## 10. Heatmap — `kind='density_heatmap'`

In [ ]:
avr.plot(
    'density_heatmap', df_dist,
    x='value', y='group',
    title='Value Density Heatmap',
    nbinsx=30,
    color_continuous_scale='Blues',
)

## 11. Strip plot — `kind='strip'`

In [ ]:
avr.plot(
    'strip', df_dist,
    x='group', y='value',
    color='group',
    title='Individual Observations by Group',
)

## 12. Overriding `DEFAULT_LAYOUT` per-call

Pass any `DEFAULT_LAYOUT` key as a kwarg to override it for that call only.

In [ ]:
# Taller figure, different template, for this call only
avr.plot(
    'bar', df_sales,
    x='month', y='revenue',
    title='Revenue (ggplot2, taller)',
    template='ggplot2',
    height=500,
    width=800,
)

## 13. Combining `avr.plot()` with organizers

Pass figures directly into `avr.tabs`, `avr.accordion`, etc.

In [ ]:
avr.tabs(
    titles=['Line', 'Bar', 'Scatter', 'Histogram'],
    contents=[
        avr.plot('line',      df_ts,    x='day',       y='price',    title='Price Trend'),
        avr.plot('bar',       df_sales, x='month',     y='revenue',  title='Monthly Revenue'),
        avr.plot('scatter',   df_sales, x='customers', y='revenue',  title='Customers vs Revenue'),
        avr.plot('histogram', df_dist,  x='value',     color='group',title='Distributions'),
    ],
)

In [ ]:
avr.accordion(
    titles=['Box Plot', 'Violin Plot', 'Strip Plot'],
    contents=[
        avr.plot('box',    df_dist, x='group', y='value', color='group', title='Box'),
        avr.plot('violin', df_dist, x='group', y='value', color='group', title='Violin', box=True),
        avr.plot('strip',  df_dist, x='group', y='value', color='group', title='Strip'),
    ],
)

In [ ]:
avr.dropdown(
    titles=['Area Chart', 'Pie Chart', 'Density Heatmap'],
    contents=[
        avr.plot('area',             df_area,  x='month', y='amount', color='metric', title='Stacked Area'),
        avr.plot('pie',              df_cat,   names='category', values='q1', title='Q1 by Category', hole=0.3),
        avr.plot('density_heatmap',  df_dist,  x='value', y='group', title='Density Heatmap', nbinsx=30),
    ],
)

## 14. Changing the global `DEFAULT_LAYOUT`

Mutate `avr.DEFAULT_LAYOUT` to change defaults for all subsequent `avr.plot()` calls.

In [ ]:
# Temporarily bump height for wider/taller charts
avr.DEFAULT_LAYOUT['height'] = 500
avr.DEFAULT_LAYOUT['width']  = 900

avr.plot('line', df_ts, x='day', y='price', title='Wider/Taller Default')

In [ ]:
# Restore defaults
avr.DEFAULT_LAYOUT.update({'height': 400, 'width': 700})
print("Restored:", avr.DEFAULT_LAYOUT)